# Lab06- Jackson Haiz

# Set Up

In [1]:
from pathlib import Path
from io import StringIO

import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from IPython.display import display

# The supplied notebook includes an embedded copy of the data.
# For a blank notebook, upload slump_test.csv or use the UCI URL.
if "SLUMP_CSV" in globals():
    raw = pd.read_csv(StringIO(SLUMP_CSV))
elif Path("slump_test.csv").is_file():
    raw = pd.read_csv("slump_test.csv")
else:
    url = (
        "https://archive.ics.uci.edu/ml/"
        "machine-learning-databases/concrete/slump/"
        "slump_test.data"
    )
    raw = pd.read_csv(url)

reference = raw.rename(columns={
    "No": "mix_id",
    "Cement": "cement",
    "Slag": "slag",
    "Fly ash": "fly_ash",
    "Water": "water",
    "SP": "superplasticizer",
    "Coarse Aggr.": "coarse_agg",
    "Fine Aggr.": "fine_agg",
    "SLUMP(cm)": "slump",
    "FLOW(cm)": "flow",
    "Compressive Strength (28-day)(Mpa)": "strength"
}).copy()

print("Original data shape:", reference.shape)
display(reference.head())
display(reference.isna().sum().rename("Original missing count"))

assert reference.shape == (103, 11)
assert not reference.isna().any().any()

# A teaching grouping, NOT an engineering standard.
reference["coarse_group"] = np.where(
    reference["coarse_agg"] >= 900,
    "At least 900",
    "Below 900"
)

fig = px.scatter(
    reference,
    x="coarse_agg",
    y="water",
    color="coarse_group",
    symbol="coarse_group",
    hover_data=["mix_id"],
    labels={
        "coarse_agg": "Coarse aggregate (kg/m^3)",
        "water": "Water (kg/m^3)",
        "coarse_group": "Coarse aggregate group"
    },
    title="Before hiding values: recorded concrete ingredients"
)

fig.show()

Original data shape: (103, 11)


,mix_id,cement,slag,fly_ash,water,superplasticizer,coarse_agg,fine_agg,slump,flow,strength
0,1,273.0,82.0,105.0,210.0,9.0,904.0,680.0,23.0,62.0,34.99
1,2,163.0,149.0,191.0,180.0,12.0,843.0,746.0,0.0,20.0,41.14
2,3,162.0,148.0,191.0,179.0,16.0,840.0,743.0,1.0,20.0,41.81
3,4,162.0,148.0,190.0,179.0,19.0,838.0,741.0,3.0,21.5,42.08
4,5,154.0,112.0,144.0,220.0,10.0,923.0,658.0,20.0,64.0,26.82


,Original missing count
mix_id,0
cement,0
slag,0
fly_ash,0
water,0
superplasticizer,0
coarse_agg,0
fine_agg,0
slump,0
flow,0


# Question 1


In [2]:
# Artificial rule (ii): probability depends on observed coarse_agg.
# Keep this seed and these probabilities unchanged.
rng = np.random.default_rng(123)
p_missing = np.where(reference["coarse_agg"] >= 900,
0.60, 0.15)
mask = rng.random(len(reference)) < p_missing
missing = reference.copy()
missing.loc[mask, "water"] = np.nan
missing["water_missing"] = mask
observed = missing.dropna(subset=["water"]).copy()
print("Artificially hidden water entries:", int(mask.sum()))
print("Complete cases retained:", len(observed))
rates = missing.groupby("coarse_group")["water_missing"].agg(
Records="size", Hidden="sum", Rate="mean"
)
rates["Missing percent"] = 100 * rates["Rate"]
display(rates[["Records", "Hidden", "Missing percent"]].round(2))
shares = pd.DataFrame({
"Reference percent": 100 * reference["coarse_group"].value_counts(
normalize=True),
"Complete-case percent": 100 * observed["coarse_group"].value_counts(
normalize=True)
})
display(shares.round(2))
fig = px.bar(
rates.reset_index(), x="coarse_group", y="Missing percent",
labels={"coarse_group": "Coarse aggregate group"},
title="Realized missingness under the artificial rule"
)
fig.show()

Artificially hidden water entries: 35
Complete cases retained: 68


,Records,Hidden,Missing percent
coarse_group,,,
At least 900,42,27,64.29
Below 900,61,8,13.11


,Reference percent,Complete-case percent
coarse_group,,
Below 900,59.22,77.94
At least 900,40.78,22.06


# Question 1.A
The amount of total source records is 103. The number of artificially hidden records is 35. The number of complete cases retained is 68.

Deliberately hiding known values for this exercise is useful so we can test how well our model predicts hidden (unseen) values.

#Question 1.B

Rule (i)- Missing Completely At Random- Missingness does not depend on anything. Values have the probability of being missing with probability .3.

Rule(ii)- Missing At Random- Missing Values depend on the set of observed responses ( in this case, above or below 900). Values above 900 coarse agg have probability .6 to be hidden, .15 probability if it falls below 900 coarse agg.

Rule(iii)- Missing not at random- Missing not at random because what determines if the variable is missing is the variable itself. Rule (iii) does not also fall into any of these other groups.

# Question 1.C

Missingness Percentage in $Coarseagg >= 900$ = 64.29 %

Missingness Percentage in $Coarseagg < 900$ = 13.11 %

I dont believe that different rates across groups will definitively prove MAR rather than MNAR. This is because the differences across groups could be attributed to something that is not observed in the data or it could be attributed to the variable itself. All in all, Different missingness percentages across groups in a real dataset does not neccesairly prove MAR rather than MNAR.

# Question 2


In [3]:
observed_mean = observed["water"].mean()

mean_data = missing.copy()
mean_data["water"] = mean_data["water"].fillna(observed_mean)

print("One value used for every gap:", round(observed_mean, 3))

frames = {
    "Reference (benchmark)": reference,
    "A: Complete cases": observed,
    "B: Mean imputation": mean_data
}

rows = []

for name, frame in frames.items():
    rows.append({
        "Dataset": name,
        "Rows": len(frame),
        "Mean water": frame["water"].mean(),
        "SD water": frame["water"].std()
    })

water_summary = pd.DataFrame(rows)
display(water_summary.round(3))

variance_ratio = mean_data["water"].var() / observed["water"].var()
identity_ratio = (len(observed) - 1) / (len(reference) - 1)

print("Completed / observed variance:", round(variance_ratio, 4))
print("(n_observed - 1) / (n - 1):", round(identity_ratio, 4))

mean_data["Entry"] = np.where(mask, "Imputed", "Observed")

fig = px.scatter(
    mean_data,
    x="coarse_agg",
    y="water",
    color="Entry",
    symbol="Entry",
    hover_data=["mix_id"],
    labels={
        "coarse_agg": "Coarse aggregate (kg/m^3)",
        "water": "Water (kg/m^3)"
    },
    title="Mean imputation: examine the imputed entries"
)

fig.show()

One value used for every gap: 201.89


,Dataset,Rows,Mean water,SD water
0,Reference (benchmark),103,197.168,20.208
1,A: Complete cases,68,201.890,18.860
2,B: Mean imputation,103,201.890,15.285


Completed / observed variance: 0.6569
(n_observed - 1) / (n - 1): 0.6569


# Question 2.A

Before Deletion: 42 Records with at least 900 $kg/mg^3$, 61 with less than 900 $kg/mg^3$

After Deletion: 15 Records with at least 900 $kg/mg^3$, 53 with less than 900 $kg/mg^3$

The group's representation changes because of rule (ii) which states that water is hidden with probability .6 when recorded coarse aggregate is at least 900 $kg/mg^3$.



# Question 2.B

1	A: Complete cases	 Obs: 68	Mean: 201.890	Std: 18.860

2	B: Mean imputation	Obs: 103	Mean: 201.890	Std: 15.285

We can see that the means across each group are exactly the same, however their standard deviations are different. This is because the imputed value model is inserting the mean in place of those missing values, which retains the same mean as the complete case deletion, but the variability shrinks because when we input those values all those values at the mean drowns out the variability in the non-imputed values. We can see in the scatterplot how those imputed values compared to the observed, and how much less variability is in the model because of them.

# Question 2.C

When we insert the median instead of the mean, the selection problem **would not** automatically be fixed. This is because the median is good at being robust, i.e resistant to outliers, but it does not automatically correct the systematic missingness. This is because missingness is contingent on an observed variable (coarse agg) which means that the observed variables left after deletion may not be fully representative of the orginal dataset if water and coarse aggregate have a defined relationship. That means when we insert the median instead of the mean, the data might be superior to outliers, but it will not fix systematic missingness because of a failure to accurately capture the relationships.

# Question 3

In [5]:
# Only observed ingredients enter the imputation model.
# Do not use strength, slump, flow, mix_id, or hidden water values.

imputer_features = [
    "cement",
    "slag",
    "fly_ash",
    "superplasticizer",
    "coarse_agg",
    "fine_agg"
]

# Fit the regression model using rows with observed water values.
imputer = LinearRegression()

imputer.fit(
    observed[imputer_features],
    observed["water"]
)

# Copy the masked dataset and impute the deliberately hidden water values.
reg_data = missing.copy()

reg_data.loc[mask, "water"] = imputer.predict(
    missing.loc[mask, imputer_features]
)

# Reveal true water values ONLY to evaluate the hidden positions.
recovery_rows = []

for name, frame in {
    "B: Mean imputation": mean_data,
    "C: Regression imputation": reg_data
}.items():

    rmse = np.sqrt(
        mean_squared_error(
            reference.loc[mask, "water"],
            frame.loc[mask, "water"]
        )
    )

    recovery_rows.append({
        "Method": name,
        "Hidden entries evaluated": int(mask.sum()),
        "Recovery RMSE (kg/m^3)": rmse
    })

recovery = pd.DataFrame(recovery_rows)

display(recovery.round(3))


# Compare dataset summary statistics across methods.
frames["C: Regression imputation"] = reg_data

rows = []

for name, frame in frames.items():
    rows.append({
        "Dataset": name,
        "Rows": len(frame),
        "Mean water": frame["water"].mean(),
        "SD water": frame["water"].std()
    })

water_summary = pd.DataFrame(rows)

display(water_summary.round(3))


# Plot regression imputation by coarse aggregate.
reg_data["Entry"] = np.where(
    mask,
    "Imputed",
    "Observed"
)

fig = px.scatter(
    reg_data,
    x="coarse_agg",
    y="water",
    color="Entry",
    symbol="Entry",
    hover_data=["mix_id"],
    labels={
        "coarse_agg": "Coarse aggregate (kg/m^3)",
        "water": "Water (kg/m^3)"
    },
    title="Regression imputation: compare with the mean-filled plot"
)

fig.show()


# Compare recovery of hidden entries for each imputation method.
# Both plots use the same axes for a fair comparison.

for name, frame in {
    "Mean imputation": mean_data,
    "Regression imputation": reg_data
}.items():

    check = pd.DataFrame({
        "Actual water": reference.loc[mask, "water"],
        "Imputed water": frame.loc[mask, "water"]
    })

    fig = px.scatter(
        check,
        x="Actual water",
        y="Imputed water",
        range_x=[150, 245],
        range_y=[150, 245],
        labels={
            "Actual water": "Actual water (kg/m^3)",
            "Imputed water": "Imputed water (kg/m^3)"
        },
        title=name + ": recovery of hidden entries"
    )

    # Add a reference line representing perfect predictions.
    fig.add_shape(
        type="line",
        x0=150,
        y0=150,
        x1=245,
        y1=245,
        line=dict(dash="dash")
    )

    fig.show()

,Method,Hidden entries evaluated,Recovery RMSE (kg/m^3)
0,B: Mean imputation,35,23.976
1,C: Regression imputation,35,4.553


,Dataset,Rows,Mean water,SD water
0,Reference (benchmark),103,197.168,20.208
1,A: Complete cases,68,201.890,18.860
2,B: Mean imputation,103,201.890,15.285
3,C: Regression imputation,103,196.921,20.548


# Question 3.A
In the imputer model, the response variable is the 'water' variable, and the predictors are the  "cement", "slag", "fly_ash", "superplasticizer", "coarse_agg", and "fine_agg" variables.  Strength is excluded because, if we include strength, there would be data leakage on the eventual response variable and that would defy the point of the second regression model, because the goal is to predict concrete strength. Including strength into the imputer model will also make the eventual regression model look stronger than it actually is because the response variable was used to predict/create a predictor, which means that the relationship will appear falsely enhanced.


# Question 3.B

| Method                | Recovery RMSE (kg/m³) |
|-----------------------|----------------------:|
| Mean imputation       | 23.976                |
| Regression imputation | 4.553                 |

I would carry forward regression imputation because it is better at reducing recovery RMSE. The recovery plot above shows the imputed values following the imputed regresion line far more closely than rhe static mean imputation.

# Question 3.C

We only score the artificially hidden rows rather than all rows because we are not predicting all rows we are only predicting rows that have been hidden. We would be scoring observed values if we scored all rows, which is pointless and would buffer the measurment of recovery RMSE for the rows that are being hidden.

Deletion removes the rows and does nothing more. There is nothing to be scored.

Unrecorded water values are unrecorded which means that we do not have $W_i$ and therefore can not estimate Recovery RMSE.

# Question 4

In [7]:
# Same later regression in every dataset:
# strength = intercept + b1 * water + b2 * cement + error.
# All metrics below are IN-SAMPLE; there is no strength test set.
strength_features = ["water", "cement"]
model_rows = []
for name, frame in frames.items():
  model = LinearRegression()
  model.fit(frame[strength_features], frame["strength"])
  fitted = model.predict(frame[strength_features])
  model_rows.append({
  "Dataset": name, "Rows": len(frame),
  "Water coefficient": model.coef_[0],
  "Cement coefficient": model.coef_[1],
  "In-sample R2": model.score(
  frame[strength_features], frame["strength"]),
  "In-sample RMSE (MPa)": np.sqrt(mean_squared_error(
  frame["strength"], fitted))
})
strength_summary = pd.DataFrame(model_rows)
display(strength_summary.round(4))

,Dataset,Rows,Water coefficient,Cement coefficient,In-sample R2,In-sample RMSE (MPa)
0,Reference (benchmark),103,-0.1439,0.0524,0.3295,6.3869
1,A: Complete cases,68,-0.0825,0.0452,0.2557,6.2173
2,B: Mean imputation,103,-0.0831,0.0462,0.2245,6.8688
3,C: Regression imputation,103,-0.1425,0.0531,0.3303,6.3830


# Question 4.A

Model C's water coeffient is the effect of a one unit increase in water content (kg/m3); on the concrete strength with ** imputed values** in place of the artificially missing values, holding all other variables constant.

The model with the highest In-Sample $R^2$ is the regression imputation model. Comparing Model A to models B and C requires care because there are different amounts of observations based on rule (ii), siphoning missing values based on coarse aggregate. The difference in $R^2$ can certaiinly be attributed to the difference in observatiosn based on this rule, which overall leads to the model being able to explain less variation in concrete strength.  

# Question 4.B

Treating deterministic imputations as known measurements understates missing data uncertainty because missing data may not follow the linear relationship that is assumed through the imputation regression model. We can easily compare the missing values to the observed values because we simulated the missingness, however in actuallity a imputation regression model is not as deterministic as we assume it to be because missing variables might not follow the linear trend of the other predictors.

Impute analyze pool is a method of imputing multiple datasets using different imputation regression models, fitting each dataset and analyzing the results, and then pooling the results from all models.

Adding random noises to the missing entries only once is not multiple imputation, because multiple imputation treats each dataset as a piece and not its own deterministic dataset and analyzes and pools off that basis. The pooling combines each of the estimates from each imputed model into one final estimate used for The actual regression. So adding random noise to the missing entries once is not mutliple imputation.

# Question 4.C

leaking safe workflow:

1. First split the data into training data and held out test sets. divide training into multiple folds. This will set up our cross validation model.

2. Predict the imputer value in each of these folds using **only the explanatory variables that will be used in the final regression model**. It is imperative to **not use** the variable we will later be predicting to avoid leakage and overall model imprecision. Then we fit strength based on the imputer water content values and cement.

3. Validate the model on the folds validation data.

4. Now predict concrete strength on a full training set, and see how well the model generalizes on a test set.


# I used CHATGPT 5.6-Luna for this lab, particular for explaining the code from the class scripts and help explain some of the details from class since Prof. Afryie was absent this week.